In [5]:
# Health_Agent_Milestone3_Complete
# Milestone 3: Synthesis & Recommendation Generation
#Integrates Milestone 1 (Ingestion) and Milestone 2 (Risk Analysis) with new Contextual & Recommendation Models.


# SETUP & INSTALLATION
import sys
import subprocess

def install_packages():
    packages = ["pdfplumber", "pytesseract", "pandas", "kaggle", "pillow"]
    for package in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import re
import pandas as pd
import numpy as np
import pdfplumber
try:
    from PIL import Image, ImageEnhance
    import pytesseract
except ImportError:
    pass
from google.colab import files

# 1. HELPER CLASSES

class Preprocessor:
    def __init__(self):
        self.column_mapping = {
            'hb': 'Haemoglobin', 'hemoglobin': 'Haemoglobin', 'hgb': 'Haemoglobin',
            'platelets': 'Platelets', 'platelet count': 'Platelets', 'plt': 'Platelets',
            'wbc': 'White Blood Cells', 'white blood cells': 'White Blood Cells',
            'rbc': 'RBC Count', 'red blood cells': 'RBC Count',
            'pcv': 'Packed Cell Volume', 'hct': 'Packed Cell Volume',
            'mcv': 'Mean Corpuscular Volume', 'mch': 'Mean Corpuscular Hemoglobin',
            'mchc': 'Mean Corpuscular Hemoglobin Concentration',
            'glucose': 'Glucose', 'fbs': 'Glucose',
            'cholesterol': 'Cholesterol'
        }

    def normalize_column_names(self, df):
        if df.empty: return df
        df.columns = [str(col).strip() for col in df.columns]
        new_cols = {}
        for col in df.columns:
            lower_col = col.lower()
            if lower_col in self.column_mapping:
                new_cols[col] = self.column_mapping[lower_col]
        return df.rename(columns=new_cols)

class DataValidator:
    def __init__(self):
        self.rules = {
            "Haemoglobin": {"min": 12.0, "max": 17.0, "unit": "g/dL"},
            "White Blood Cells": {"min": 4000, "max": 11000, "unit": "/cumm"},
            "RBC Count": {"min": 3.8, "max": 5.8, "unit": "mill/cumm"},
            "Platelets": {"min": 150000, "max": 450000, "unit": "/cumm"},
            "Packed Cell Volume": {"min": 36, "max": 50, "unit": "%"},
            "Mean Corpuscular Volume": {"min": 80, "max": 100, "unit": "fL"},
            "Mean Corpuscular Hemoglobin": {"min": 27, "max": 32, "unit": "pg"},
            "Mean Corpuscular Hemoglobin Concentration": {"min": 32, "max": 36, "unit": "g/dL"},
            "Glucose": {"min": 70, "max": 140, "unit": "mg/dL"},
            "Cholesterol": {"min": 0, "max": 200, "unit": "mg/dL"}
        }

class CommonInterpreter:
    def read_file(self, file_path):
        if file_path.lower().endswith('.csv'): return pd.read_csv(file_path)
        elif file_path.lower().endswith('.pdf'): return self.read_pdf(file_path)
        elif file_path.lower().endswith(('.jpg', '.png', '.jpeg')): return self.read_image(file_path)
        return pd.DataFrame()

    def read_pdf(self, file_path):
        all_data = []
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    parsed = self.parse_text_to_data(text)
                    if parsed: all_data.append(pd.DataFrame([parsed]))
        return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

    def read_image(self, file_path):
        try:
            img = Image.open(file_path).convert('L')
            img = ImageEnhance.Contrast(img).enhance(2.0)
            img = img.point(lambda x: 255 if x > 200 else 0, mode='1')
            text = pytesseract.image_to_string(img)
            parsed = self.parse_text_to_data(text)
            return pd.DataFrame([parsed]) if parsed else pd.DataFrame()
        except: return pd.DataFrame()

    def parse_text_to_data(self, text):
        data = {}
        patterns = {
            "Haemoglobin": [r"(?i)(?:Haemoglobin|Hb)[\s\:\-]+(\d+\.?\d*)"],
            "White Blood Cells": [r"(?i)(?:WBC|White\s+Blood\s+Cells?)[\s\:\-]+(\d+(?:[\.,]\d+)?)"],
            "RBC Count": [r"(?i)(?:RBC|Red\s+Blood\s+Cells?)[\s\:\-]+(\d+\.?\d*)"],
            "Platelets": [r"(?i)(?:Platelets|PLT)[\s\:\-]+(\d+(?:\.?\d+)?)"],
            "Packed Cell Volume": [r"(?i)(?:PCV|HCT)[\s\:\-]+(\d+\.?\d*)"],
            "Mean Corpuscular Volume": [r"(?i)MCV[\s\:\-]+(\d+\.?\d*)"],
            "Mean Corpuscular Hemoglobin": [r"(?i)MCH\b[\s\:\-]+(\d+\.?\d*)"],
            "Mean Corpuscular Hemoglobin Concentration": [r"(?i)MCHC[\s\:\-]+(\d+\.?\d*)"],
            "Glucose": [r"(?i)(?:Glucose|FBS)[\s\:\-]+(\d+\.?\d*)"],
            "Cholesterol": [r"(?i)(?:Cholesterol)[\s\:\-]+(\d+\.?\d*)"]
        }
        for param, regex_list in patterns.items():
            for pattern in regex_list:
                match = re.search(pattern, text)
                if match:
                    try: data[param] = float(match.group(1).replace(',', ''))
                    except: continue
        return data


# 2. INTELLIGENCE MODELS (M3 Scope)

class PatientContextExtractor:
    def extract(self, text):
        context = {'age': 30, 'gender': 'Female'} # Default fallback
        if not text: return context

        age_match = re.search(r"(?i)(?:Age|Yrs?|Age\/Gender)[\s\:\-\/]+(\d+)", text)
        if age_match: context['age'] = int(age_match.group(1))

        if re.search(r"(?i)\b(?:Female|Woman|Mrs\.|Ms\.)\b", text): context['gender'] = "Female"
        elif re.search(r"(?i)\b(?:Male|Man|Mr\.)\b", text): context['gender'] = "Male"
        elif re.search(r"(?i)(?:Sex|Gender)[\s\:\-\/]+F\b", text): context['gender'] = "Female"
        elif re.search(r"(?i)(?:Sex|Gender)[\s\:\-\/]+M\b", text): context['gender'] = "Male"
        elif re.search(r"(?i)\/F\b", text): context['gender'] = "Female"

        return context

class ContextualAnalysisModel:
    def adjust_rules(self, base_rules, age, gender):
        adj_rules = {k: v.copy() for k, v in base_rules.items()}
        if 'Haemoglobin' in adj_rules:
            if gender.lower().startswith('f'):
                adj_rules['Haemoglobin'].update({'min': 12.0, 'max': 15.5})
            elif gender.lower().startswith('m'):
                adj_rules['Haemoglobin'].update({'min': 13.5, 'max': 17.5})
        if 'Glucose' in adj_rules and age > 60:
            adj_rules['Glucose']['max'] = 140
        return adj_rules

class BiomarkerCorrelationEngine:
    def __init__(self):
        self.patterns = {
            "metabolic_syndrome": {
                "name": "Metabolic Syndrome",
                "markers": ["Glucose", "Cholesterol"],
                "required": 2,
                "significance": "Insulin resistance, cardiovascular risk"
            },
            "anemia": {
                "name": "Anemia",
                "markers": ["Haemoglobin", "RBC Count"],
                "required": 2,
                "significance": "Low oxygen-carrying capacity"
            },
            "infection": {
                "name": "Active Infection",
                "markers": ["White Blood Cells"],
                "required": 1,
                "significance": "Immune response activation"
            }
        }

    def analyze(self, biomarkers, rules):
        found = []
        deviations = {}
        for m, v in biomarkers.items():
            if m in rules:
                rule = rules[m]
                if v < rule['min'] or v > rule['max']:
                    deviations[m] = v

        for pid, cfg in self.patterns.items():
            matches = [m for m in cfg['markers'] if m in deviations]
            if len(matches) >= cfg['required']:
                found.append({
                    "Pattern": cfg['name'],
                    "Significance": cfg['significance'],
                    "Evidence": matches
                })
        return found

class RecommendationGenerator:
    def __init__(self):
        self.kb = {
            "Metabolic Syndrome": {
                "Diet": "Low-glycemic index foods. Reduce sugar.",
                "Lifestyle": "150 mins aerobic exercise/week."
            },
            "Anemia": {
                "Diet": "Iron-rich foods (Spinach, Red Meat).",
                "Lifestyle": "Rest and avoid strenuous activity."
            },
             "Active Infection": {
                "Diet": "Hydrate well. Eat nutritious, light meals.",
                "Lifestyle": "Rest to support immune response. Monitor fever."
            }
        }

    def generate_report(self, risks, context):
        advice = []
        if not risks:
            advice.append("✅ HEALTHY STATUS: No complex risk patterns detected. Maintain current lifestyle.")

        for r in risks:
            name = r['Pattern']
            item = self.kb.get(name, {"Diet": "Consult Doctor", "Lifestyle": "Consult Doctor"})

            lifestyle = item['Lifestyle']
            if context['age'] > 60 and "aerobic" in lifestyle:
                lifestyle = "Low-impact activity (Walking/Swimming)"

            advice.append(f"👉 {name.upper()}:")
            advice.append(f"   Diet: {item['Diet']}")
            advice.append(f"   Lifestyle: {lifestyle}")

        return "\n".join(advice)

class HealthAgent:
    def __init__(self):
        self.context_extractor = PatientContextExtractor()
        self.context_model = ContextualAnalysisModel()
        self.risk_engine = BiomarkerCorrelationEngine()
        self.recommender = RecommendationGenerator()
        self.validator = DataValidator()
        self.base_rules = self.validator.rules

    def process_data(self, df, raw_text):
        context = self.context_extractor.extract(raw_text)
        print(f"   🤖 AI INFERENCE: Detected Patient as {context['age']} Y, {context['gender']}")

        active_rules = self.context_model.adjust_rules(self.base_rules, context['age'], context['gender'])

        biomarkers = {}
        if not df.empty:
             for col in df.columns:
                 if col in active_rules:
                      vals = pd.to_numeric(df[col], errors='coerce').dropna()
                      if not vals.empty:
                          val = float(vals.iloc[0])
                          biomarkers[col] = val

        print(f"   🔎 MODEL 1: Extracted {len(biomarkers)} Clinical Parameters")

        for k, v in biomarkers.items():
            rule = active_rules[k]
            status = "Normal"
            if v < rule['min']: status = f"LOW (<{rule['min']})"
            elif v > rule['max']: status = f"HIGH (>{rule['max']})"
            print(f"      ▪ {k}: {v} -> {status}")

        risks = self.risk_engine.analyze(biomarkers, active_rules)
        if risks:
            print(f"   🧬 MODEL 2: Detected {len(risks)} Risk Patterns")
        else:
            print("   ✅ MODEL 2: No High-Risk Patterns Detected")

        advice_text = self.recommender.generate_report(risks, context)

        return {
            "patient": context,
            "biomarkers": biomarkers,
            "risks": risks,
            "advice": advice_text
        }

# 3. MAIN EXECUTION
if __name__ == "__main__":
    print("--- 🩺 HEALTH AGENT START ---")

    interpreter = CommonInterpreter()
    preprocessor = Preprocessor()
    agent = HealthAgent()

    print("\nPlease upload your Medical Reports (PDF/Image)...")
    uploaded = files.upload()

    if uploaded:
        for filename in uploaded.keys():
            print(f"\n" + "="*50)
            print(f"📂 PROCESSING FILE: {filename}")
            print("="*50)

            try:
                raw_text = ""
                if filename.lower().endswith('.pdf'):
                    with pdfplumber.open(filename) as pdf:
                        for p in pdf.pages:
                            raw_text += (p.extract_text() or "") + "\n"

                df = interpreter.read_file(filename)
                df = preprocessor.normalize_column_names(df)

                if df.empty:
                    print("⚠️ Warning: Could not extract structured tables.")

                result = agent.process_data(df, raw_text)

                print("\n--- 📝 FINAL REPORT ---")
                print(result['advice'])
                print("-" * 30)

            except Exception as e:
                print(f"❌ Error processing {filename}: {e}")
    else:
        print("No files uploaded.")


--- 🩺 HEALTH AGENT START ---

Please upload your Medical Reports (PDF/Image)...


Saving BDCBC7196_Hematology_Dataset.csv to BDCBC7196_Hematology_Dataset.csv
Saving Blood_report_pdf_6.pdf to Blood_report_pdf_6 (4).pdf

📂 PROCESSING FILE: BDCBC7196_Hematology_Dataset.csv
   🤖 AI INFERENCE: Detected Patient as 30 Y, Female
   🔎 MODEL 1: Extracted 8 Clinical Parameters
      ▪ Haemoglobin: 12.1 -> Normal
      ▪ RBC Count: 4.25 -> Normal
      ▪ White Blood Cells: 12300.0 -> HIGH (>11000)
      ▪ Platelets: 404000.0 -> Normal
      ▪ Packed Cell Volume: 36.2 -> Normal
      ▪ Mean Corpuscular Volume: 85.2 -> Normal
      ▪ Mean Corpuscular Hemoglobin: 28.4 -> Normal
      ▪ Mean Corpuscular Hemoglobin Concentration: 33.4 -> Normal
   🧬 MODEL 2: Detected 1 Risk Patterns

--- 📝 FINAL REPORT ---
👉 ACTIVE INFECTION:
   Diet: Hydrate well. Eat nutritious, light meals.
   Lifestyle: Rest to support immune response. Monitor fever.
------------------------------

📂 PROCESSING FILE: Blood_report_pdf_6 (4).pdf
   🤖 AI INFERENCE: Detected Patient as 54 Y, Female
   🔎 MODEL 1: Ext